# k-Nearest Neighbours on MNIST

**Goal:** apply the same $k$-NN workflow to real handwritten digits. MNIST contains
70,000 grayscale images of size $28 \times 28$. Flattened, each image is a vector of 784
pixel intensities, and the task is 10-class classification.

$k$-NN does surprisingly well on MNIST --- around 97% test accuracy with a few thousand
training images --- but prediction is slow because every query is compared with every
stored training point.

**Practical note.** The notebook tries to download MNIST from OpenML (about 70 MB). If
that fails, for example when you are offline, it falls back automatically to
scikit-learn's 8x8 `digits` dataset. Every later cell still runs; only the image
resolution and accuracy change. To skip the download entirely, set
`DOWNLOAD_MNIST = False` in the setup cell or start Jupyter with `KNN_FORCE_DIGITS=1`.

## Setup

The setup is split into two cells. The first imports the lightweight libraries and sets
the runtime options, so it finishes immediately. The second imports matplotlib and
scikit-learn; on a fresh machine it can take 10--30 seconds because matplotlib builds its
font cache once.

In [ ]:
import os
from time import perf_counter

import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set DOWNLOAD_MNIST = False to skip the OpenML download; the environment
# variable KNN_FORCE_DIGITS=1 has the same effect.
DOWNLOAD_MNIST = os.environ.get("KNN_FORCE_DIGITS", "0") != "1"

print("Core setup complete.")

The next cell is the expensive one on a first run. It imports matplotlib (which may spend
10--30 seconds building its font cache) and scikit-learn. The progress messages show
which import is running.

In [ ]:
print("Importing matplotlib (first run may build the font cache)...", flush=True)
import matplotlib.pyplot as plt

print("Importing scikit-learn...", flush=True)
from sklearn.datasets import fetch_openml, load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Scientific libraries ready.")

### Reusable helpers

The same helpers as in the Iris notebook, plus a gallery function that can display
grayscale images from flattened vectors.

In [ ]:
def build_knn_pipeline(k, n_components=None, random_state=RANDOM_STATE):
    # Standardise, optionally reduce with PCA, then classify with k-NN.
    steps = [("scaler", StandardScaler())]
    if n_components is not None:
        steps.append(("pca", PCA(n_components=n_components, random_state=random_state)))
    steps.append(("knn", KNeighborsClassifier(n_neighbors=int(k), n_jobs=-1)))
    return Pipeline(steps)


def tune_k(X, y, ks, cv_splits=3, n_components=None, random_state=RANDOM_STATE):
    # Cross-validated accuracy for every candidate k.
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    rows = []
    for k in ks:
        model = build_knn_pipeline(k, n_components=n_components, random_state=random_state)
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
        rows.append(
            {"k": int(k), "mean": float(scores.mean()), "std": float(scores.std())}
        )
    return rows


def best_row(rows):
    return max(rows, key=lambda row: row["mean"])


def plot_k_curve(rows, title):
    ks = [row["k"] for row in rows]
    means = np.array([row["mean"] for row in rows])
    stds = np.array([row["std"] for row in rows])
    j = int(np.argmax(means))

    fig, ax = plt.subplots()
    ax.errorbar(ks, means, yerr=stds, marker="o", capsize=3)
    ax.axvline(ks[j], color="k", linestyle=":", label=f"best k = {ks[j]}")
    ax.set_xlabel("$k$")
    ax.set_ylabel("Cross-validated accuracy")
    ax.set_title(title)
    ax.legend()
    plt.show()


def fit_and_time(model, X_train, y_train, X_test, y_test):
    # Fit, predict, and report accuracy plus per-query prediction time.
    model.fit(X_train, y_train)
    model.predict(X_test[:5])  # warm-up
    start = perf_counter()
    y_pred = model.predict(X_test)
    elapsed = perf_counter() - start
    return {
        "model": model,
        "y_pred": y_pred,
        "acc": accuracy_score(y_test, y_pred),
        "seconds": elapsed,
        "ms_per_query": 1000.0 * elapsed / len(X_test),
    }


def plot_image_gallery(images, labels, class_names, shape, n=12, title=""):
    # Show a grid of grayscale or RGB images from flattened vectors.
    n = min(n, len(images))
    cols = 6
    n_rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 1.7, n_rows * 2.0))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, label in zip(axes.ravel(), images[:n], labels[:n]):
        if img.ndim == 3:
            ax.imshow(img)
        else:
            ax.imshow(img.reshape(shape), cmap="gray")
        ax.set_title(str(class_names[label]), fontsize=9)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## 1. A loader that works offline

`fetch_openml("mnist_784")` needs a network connection and downloads about 70 MB. The
function below tries it, and if anything goes wrong it falls back to scikit-learn's
built-in 8x8 `digits` dataset.

To keep the notebook fast we also take a stratified subsample. The same code scales to
the full dataset if you are patient.

In [ ]:
def load_mnist(n_train=6000, n_test=1200, random_state=42, force_digits=False):
    # Load MNIST from OpenML, falling back to the 8x8 digits dataset when offline.
    X = None
    source = "MNIST (OpenML mnist_784)"
    shape = (28, 28)

    if not force_digits:
        try:
            print("Fetching mnist_784 from OpenML (~70 MB). "
                  "This can take a few minutes...")
            start = perf_counter()
            mnist = fetch_openml(
                "mnist_784", version=1, as_frame=False, parser="auto",
                n_retries=1,
            )
            X = np.asarray(mnist.data, dtype=np.float32)
            y = np.asarray(mnist.target, dtype=np.int64)
            print(f"Downloaded MNIST in {perf_counter() - start:.1f} s")
        except Exception as exc:
            print(f"MNIST download failed ({type(exc).__name__}); using load_digits.")
            print("Tip: set DOWNLOAD_MNIST = False in the setup cell to skip this next time.")

    if X is None:
        digits = load_digits()
        X = np.asarray(digits.data, dtype=np.float32)
        y = np.asarray(digits.target, dtype=np.int64)
        source = "scikit-learn digits (8x8 fallback)"
        shape = (8, 8)

    requested = n_train + n_test
    n_total = min(requested, len(X))
    if n_total >= len(X):
        X_pool, y_pool = X, y
    else:
        X_pool, _, y_pool, _ = train_test_split(
            X, y, train_size=n_total, stratify=y, random_state=random_state
        )

    test_fraction = n_test / requested if n_total == requested else 0.20
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_pool, y_pool, test_size=test_fraction,
        stratify=y_pool, random_state=random_state,
    )
    return X_tr, X_te, y_tr, y_te, source, shape


X_mn_train, X_mn_test, y_mn_train, y_mn_test, mnist_source, mnist_shape = load_mnist(
    n_train=6000,
    n_test=1200,
    random_state=RANDOM_STATE,
    force_digits=not DOWNLOAD_MNIST,
)

print(f"Source: {mnist_source}")
print(f"Image shape: {mnist_shape}")
print(f"Train: {X_mn_train.shape}  Test: {X_mn_test.shape}")
print(f"Pixel range: [{X_mn_train.min():.0f}, {X_mn_train.max():.0f}]")
print("Class counts in train:", np.bincount(y_mn_train))

If the download succeeded you should see 28x28 images and pixel values in $[0, 255]$.
If the fallback ran, the images are 8x8 and the pixel range is smaller. Both cases work
with the rest of the notebook.

## 2. Inspect a few images

Each digit varies in slant, thickness, and position. $k$-NN does not know that pixels are
spatially arranged: it treats the 784 values as independent coordinates.

In [ ]:
class_names_mnist = [str(i) for i in np.unique(y_mn_train)]
mnist_images = (X_mn_train[:12] / 255.0).reshape(-1, *mnist_shape)
plot_image_gallery(
    mnist_images, y_mn_train[:12], class_names_mnist, mnist_shape,
    n=12, title=f"Training samples: {mnist_source}",
)

Pixel-space distance is a crude notion of digit similarity, which is exactly why a
convolutional network eventually wins. Still, neighbour voting works well because similar
digits tend to have similar ink patterns.

## 3. Choose $k$

Cross-validating on the full training set would be slow. We tune on a stratified
2,000-image subset and then refit the chosen model on all available training images.
This is a standard compromise: model selection on a manageable sample, final fit on more
data.

In [ ]:
n_tune = min(2000, len(X_mn_train))
if n_tune < len(X_mn_train):
    X_mn_tune, _, y_mn_tune, _ = train_test_split(
        X_mn_train, y_mn_train, train_size=n_tune,
        stratify=y_mn_train, random_state=RANDOM_STATE,
    )
else:
    X_mn_tune, y_mn_tune = X_mn_train, y_mn_train
print(f"Tuning on {len(X_mn_tune)} images with 3-fold CV")

ks_mnist = [1, 3, 5, 7, 9, 15]
rows_mnist = tune_k(X_mn_tune, y_mn_tune, ks_mnist, cv_splits=3)
best_mnist = best_row(rows_mnist)

plot_k_curve(rows_mnist, "MNIST: choosing k on a stratified subset")

for row in rows_mnist:
    print(f"k = {row['k']:2d}   CV accuracy = {row['mean']:.4f} +/- {row['std']:.4f}")
print(f"\nSelected k = {best_mnist['k']}")

MNIST is high-dimensional but not *statistically* high-dimensional in the way random
Gaussian data are: the digits lie near a low-dimensional manifold, so neighbours remain
meaningful. Small odd values of $k$ (3 or 5) usually win. $k=1$ is often close behind but
has no smoothing and is sensitive to label noise.

## 4. Raw pixels versus PCA

We compare two models with the same $k$:

- **raw pixels:** 784 features (or 64 for the fallback), brute-force neighbour search;
- **PCA-50:** project onto the 50 leading principal components, then search.

PCA is fitted on the training set only, inside the pipeline. It often *improves* accuracy
slightly by denoising, and it makes prediction much faster because distances are computed
in 50 dimensions instead of 784.

In [ ]:
best_k_mnist = best_mnist["k"]

mnist_raw = fit_and_time(
    build_knn_pipeline(best_k_mnist),
    X_mn_train, y_mn_train, X_mn_test, y_mn_test,
)
mnist_pca = fit_and_time(
    build_knn_pipeline(best_k_mnist, n_components=50),
    X_mn_train, y_mn_train, X_mn_test, y_mn_test,
)

print(f"{'model':<22}{'test accuracy':>15}{'seconds':>10}{'ms/query':>10}")
print("-" * 57)
for name, result in [("raw pixels", mnist_raw), ("PCA-50", mnist_pca)]:
    print(f"{name:<22}{result['acc']:>15.4f}{result['seconds']:>10.2f}"
          f"{result['ms_per_query']:>10.3f}")

raw_queries = len(X_mn_test) * len(X_mn_train) * X_mn_train.shape[1]
pca_queries = len(X_mn_test) * len(X_mn_train) * 50


def format_operations(value):
    if value >= 1e9:
        return f"{value / 1e9:.1f} billion"
    return f"{value / 1e6:.0f} million"


print(f"\nRough distance operations, raw: {format_operations(raw_queries)}")
print(f"Rough distance operations, PCA: {format_operations(pca_queries)}")

For full MNIST the speed-up is roughly the ratio of dimensions, $784/50 \approx 16$. On
the 8x8 fallback the gain is small because there are only 64 features to begin with. On a
modern BLAS implementation the constant factor is not exactly the dimension ratio, but
the order of magnitude is clear. If accuracy holds up, PCA is almost free performance.

## 5. Confusion matrix and errors

We diagnose the better of the two models on the test set.

In [ ]:
if mnist_pca["acc"] >= mnist_raw["acc"]:
    mnist_best, mnist_best_name = mnist_pca, "PCA-50 + k-NN"
else:
    mnist_best, mnist_best_name = mnist_raw, "raw pixels + k-NN"

mnist_pred = mnist_best["y_pred"]
print(f"Diagnosing: {mnist_best_name}")
print(f"Test accuracy: {mnist_best['acc']:.4f}\n")
print(classification_report(
    y_mn_test, mnist_pred,
    target_names=class_names_mnist, digits=3, zero_division=0,
))

cm_mnist = confusion_matrix(y_mn_test, mnist_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm_mnist, display_labels=class_names_mnist).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title(f"MNIST confusion matrix ({mnist_best_name}, k = {best_k_mnist})")
plt.show()

In [ ]:
wrong = np.where(mnist_pred != y_mn_test)[0]
print(f"Misclassified: {len(wrong)} / {len(y_mn_test)} test images")

if len(wrong):
    n_show = min(8, len(wrong))
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, idx in zip(axes.ravel(), wrong[:n_show]):
        ax.imshow(X_mn_test[idx].reshape(mnist_shape), cmap="gray")
        ax.set_title(
            f"true: {class_names_mnist[y_mn_test[idx]]}   "
            f"pred: {class_names_mnist[mnist_pred[idx]]}",
            fontsize=9, color="darkred",
        )
    fig.suptitle("Typical k-NN errors on MNIST")
    fig.tight_layout()
    plt.show()
else:
    print("No errors on this test subsample.")

Most errors confuse visually similar digits: 4/9, 3/5, 7/1, and some 8/9. A human would
often make the same mistake on the worst examples. This is a useful reminder that 97%
accuracy does not mean the model has learned the concept of a digit --- it has learned a
local similarity metric.

**Takeaways**

- $k$-NN is a strong baseline on small images;
- tune $k$ on a subset, then refit on more data;
- PCA reduces prediction cost dramatically and may even help accuracy;
- per-class metrics matter: some digits are systematically harder.

## What to try next

1. **Sample size.** Change `n_train` in `load_mnist` from 6,000 to 2,000 and to 12,000.
   Plot test accuracy and prediction time against training size.
2. **$k=1$.** Confirm that the training accuracy of a 1-NN model is exactly 1.0. Explain
   why, then compare its test accuracy with $k=5$.
3. **Distance weighting.** Add `weights="distance"` to `build_knn_pipeline` and rerun.
   Where does it help?
4. **Cosine distance.** Compare `metric="euclidean"` with `metric="cosine"`. Explain the
   difference for pixel vectors.
5. **PCA variance.** Plot the cumulative explained variance. How many components are
   needed for 90%? How does accuracy change if you use that many?
6. **Denoising.** Try `PCA(n_components=50, whiten=True)` and compare with the default.
7. **Full MNIST.** Run the tuned pipeline on all 60,000 training images. Measure fit time
   and prediction time, and explain why brute force becomes impractical for interactive
   use.

## Further reading

- T. Cover and P. Hart, "Nearest Neighbor Pattern Classification", *IEEE Transactions on
  Information Theory*, 1967.
- Y. LeCun, L. Bottou, Y. Bengio, and P. Haffner, "Gradient-based learning applied to
  document recognition", *Proceedings of the IEEE*, 1998.
- scikit-learn user guide, *Nearest Neighbors*.